# Phase 4 — Machine Learning: Match Outcome Predictor

## The ML Problem

We frame this as a **multi-class classification** problem:

> Given features describing both teams before a match, predict whether the result will be **Win (W)**, **Draw (D)**, or **Loss (L)** from the home team's perspective.

This is classification (not regression) because outcomes are discrete categories, not continuous numbers.

## Why XGBoost?

| Model | Reason for/against |
|-------|-------------------|
| Logistic Regression | Too simple — assumes linear decision boundaries. Football is non-linear. |
| Neural Network | Overkill for tabular data with 14 features and 40k rows. Hard to interpret. |
| Random Forest | Good baseline, but XGBoost almost always beats it on tabular data. |
| **XGBoost** | State-of-the-art for tabular classification. Handles missing values natively. Gives feature importance. Proven in sports prediction literature. |

XGBoost builds an ensemble of decision trees where each new tree **corrects the errors** of the previous ones (gradient boosting). This makes it excellent at learning complex patterns like "Brazil tends to win when their last-10 win rate > 0.7 AND the opponent's average goals conceded is > 1.5."

## Features → Prediction

```
home_win_rate_last10        = How hot is the home team recently?
away_win_rate_last10        = How hot is the away team recently?
home_avg_goals_scored_last20 = Offensive firepower
away_avg_goals_conceded_last20 = Away team's defensive fragility
home_goal_diff_trend        = Is home team improving or declining?
home_ranking_proxy          = Overall team strength
h2h_win_rate                = Historical dominance vs THIS opponent
h2h_total                   = How reliable is the H2H signal?
        ↓
    XGBoost
        ↓
P(Win)=0.52, P(Draw)=0.28, P(Loss)=0.20  → Predicted: Win (52% confidence)
```

## Data Flow
- **Training**: `match_features` table — ~40,000 historical matches with known outcomes
- **Inference**: `mart_predictions_input` dbt table — 48 WC 2026 group stage fixtures
- **Output**: `wc2026_predictions` table in PostgreSQL + `models/xgb_fifa_model.pkl`

In [ ]:
# ============================================================
# Cell 2 — Load training data and inference data from PostgreSQL
# ============================================================
import os
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")  # Suppress sklearn deprecation noise

# Load secrets from .env
load_dotenv(dotenv_path=Path("..") / ".env")
DB_URL = os.getenv("DATABASE_URL")

if not DB_URL:
    raise EnvironmentError("DATABASE_URL not found in .env")

engine = create_engine(DB_URL, echo=False)

# ============================================================
# TRAINING DATA — match_features (PySpark output, Phase 2)
# Contains historical matches WITH known outcomes (the 'result' column)
# This is the supervised learning dataset.
# ============================================================
print("Loading training data from match_features table...")

df_train_raw = pd.read_sql("""
    SELECT
        date,
        home_team,
        away_team,
        result,                          -- Target variable (W/D/L)
        -- Home team features
        home_win_rate_last10,
        home_draw_rate_last10,
        home_avg_goals_scored_last20,
        home_avg_goals_conceded_last20,
        home_avg_goal_diff_last10,
        home_goal_diff_trend,
        home_ranking_proxy,
        home_ranking_change,
        home_matches_available,
        -- Away team features
        away_win_rate_last10,
        away_draw_rate_last10,
        away_avg_goals_scored_last20,
        away_avg_goals_conceded_last20,
        away_avg_goal_diff_last10,
        away_goal_diff_trend,
        away_ranking_proxy,
        away_ranking_change,
        away_matches_available,
        -- Head-to-head features
        h2h_team_a_win_rate,
        h2h_total
    FROM match_features
    WHERE result IS NOT NULL           -- Must have a known outcome
      AND date >= '1990-01-01'         -- Modern era only; pre-1990 football is too different
    ORDER BY date
""", engine)

print(f"Training rows loaded: {len(df_train_raw):,}")

# ============================================================
# INFERENCE DATA — mart_predictions_input (dbt output, Phase 3)
# Contains the 48 WC 2026 fixtures WITHOUT outcomes (future matches)
# We need to rename columns to match training features.
# ============================================================
print("\nLoading 2026 WC fixture data from mart_predictions_input...")

# Try dbt mart schema first, then fallback to public schema
for schema in ["marts", "public"]:
    try:
        df_fixtures_raw = pd.read_sql(
            f"SELECT * FROM {schema}.mart_predictions_input", engine
        )
        print(f"  Found in schema: {schema}")
        break
    except Exception:
        continue
else:
    # Fallback: use the seed table directly (if dbt hasn't run yet)
    print("  mart_predictions_input not found. Using seed fixtures + computing features manually.")
    df_fixtures_raw = pd.read_sql("""
        SELECT fixture_id, group_name, home_team, away_team, match_date
        FROM wc2026_fixtures
    """, engine)

print(f"Fixtures loaded: {len(df_fixtures_raw):,}")

# ---- Preview both datasets ----
print("\nTraining data sample:")
display(df_train_raw.head(3))

print("\nFixture data columns:")
print(list(df_fixtures_raw.columns))

engine.dispose()

## Feature Selection & Preprocessing

### Why these specific features?

We use features that:
1. **Exist in both training and inference data** — we can't train on a feature we can't compute for the 2026 fixtures
2. **Have predictive signal** — verified by domain knowledge (recent form predicts outcomes)
3. **Are numeric** — XGBoost works with numbers; we encode W/D/L → 0/1/2

### Train/Test Split Strategy

We use **stratified split** (80% train, 20% test) which ensures the W/D/L ratio is the same in both sets. Without stratification, the test set might accidentally contain all draws, making accuracy misleadingly high or low.

We do **NOT** use time-based split (train on pre-2020, test on post-2020) because:
- Our rolling features already prevent data leakage
- Football tactics change slowly; a random split is a fairer evaluation

In [ ]:
# ============================================================
# Cell 3 — Feature selection, preprocessing, train/test split
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# ---- Define the canonical feature set ----
# These columns exist in match_features (training) AND
# can be mapped from mart_predictions_input (inference)
FEATURE_COLS = [
    "home_win_rate_last10",          # Short-term home form
    "away_win_rate_last10",          # Short-term away form
    "home_draw_rate_last10",         # Draw tendency (high = cagey team)
    "away_draw_rate_last10",
    "home_avg_goals_scored_last20",  # Offensive capability
    "away_avg_goals_scored_last20",
    "home_avg_goals_conceded_last20",# Defensive solidity
    "away_avg_goals_conceded_last20",
    "home_avg_goal_diff_last10",     # Net dominance
    "away_avg_goal_diff_last10",
    "home_goal_diff_trend",          # Improving or declining?
    "away_goal_diff_trend",
    "home_ranking_proxy",            # Overall team strength
    "away_ranking_proxy",
    "home_ranking_change",           # Ranking momentum
    "away_ranking_change",
    "h2h_team_a_win_rate",           # Historical H2H dominance
    "h2h_total",                     # H2H sample size (reliability signal)
]
TARGET_COL = "result"

# ---- Encode target variable ----
# XGBoost requires integer class labels (not strings).
# We encode: W=0, D=1, L=2
# Using a fixed mapping (not LabelEncoder) so the meaning is always consistent.
LABEL_MAP     = {"W": 0, "D": 1, "L": 2}
LABEL_MAP_INV = {0: "W", 1: "D", 2: "L"}
LABEL_NAMES   = ["W (Home Win)", "D (Draw)", "L (Home Loss)"]

# ---- Prepare training DataFrame ----
df_train = df_train_raw[FEATURE_COLS + [TARGET_COL]].copy()

# Fill any remaining nulls with column median
# (Some older matches may have sparse h2h history)
for col in FEATURE_COLS:
    if df_train[col].isnull().any():
        median_val = df_train[col].median()
        df_train[col] = df_train[col].fillna(median_val)
        print(f"  Filled {df_train[col].isnull().sum()} nulls in {col} with median={median_val:.3f}")

X = df_train[FEATURE_COLS]
y = df_train[TARGET_COL].map(LABEL_MAP)  # 'W' → 0, 'D' → 1, 'L' → 2

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution in full dataset:")
for label, count in y.value_counts().sort_index().items():
    print(f"  {LABEL_MAP_INV[label]} ({label}): {count:,} ({count/len(y)*100:.1f}%)")

# ---- Train/test split (80/20, stratified) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,     # Fixed seed for reproducibility
    stratify=y           # Maintain W/D/L proportions in both splits
)

print(f"\nTrain size: {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test size : {len(X_test):,} ({len(X_test)/len(X)*100:.0f}%)")

# ---- Scale features ----
# XGBoost doesn't strictly need scaling (tree-based models are scale-invariant),
# but scaling helps if we compare with other models later and ensures
# feature contribution calculations are on the same scale.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on train ONLY (prevent leakage)
X_test_scaled  = scaler.transform(X_test)        # Apply same transform to test

# Store feature means/stds for later (used in feature contribution cell)
feature_means = scaler.mean_
feature_stds  = scaler.scale_

print("\nFeature scaling complete (StandardScaler fit on train set only).")
print("\nFeature summary (training set):")
display(X_train.describe().round(3))

## Training the XGBoost Classifier

### Hyperparameters explained

| Parameter | Value | What it controls |
|-----------|-------|------------------|
| `n_estimators` | 300 | Number of trees. More = better up to a point, then overfitting. |
| `max_depth` | 4 | Depth of each tree. Deeper = more complex patterns = risk of overfit. |
| `learning_rate` | 0.05 | How much each tree corrects the previous. Slower = more careful. |
| `subsample` | 0.8 | Train each tree on 80% of rows (random). Adds diversity. |
| `colsample_bytree` | 0.8 | Train each tree on 80% of features. Prevents over-reliance on one feature. |
| `early_stopping_rounds` | 20 | Stop if test loss hasn't improved in 20 rounds. Prevents overfit. |
| `objective` | `multi:softprob` | Multi-class output as probabilities (not just class label). |

### What the metrics mean
- **Accuracy**: fraction of predictions that match the actual outcome
- **Precision (per class)**: of all matches predicted as W, what % were actually W?
- **Recall (per class)**: of all actual W matches, what % did the model correctly predict as W?
- **F1**: harmonic mean of precision and recall (balanced metric)

> Expected accuracy: 50–58%. This is typical for football prediction — the sport is inherently unpredictable. A model above 55% is considered good in the academic literature.

In [ ]:
# ============================================================
# Cell 4 — Train XGBoost, evaluate performance
# ============================================================
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
import plotly.graph_objects as go
import plotly.figure_factory as ff

print(f"XGBoost version: {xgb.__version__}")

# ---- Train XGBoost multi-class classifier ----
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,      # Minimum samples per leaf (prevents tiny noisy leaves)
    gamma=0.1,               # Minimum loss reduction to make a split (regularization)
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",  # Multi-class log loss
    early_stopping_rounds=20,# Stop if no improvement for 20 rounds
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,               # Use all available CPU cores
)

# Fit with eval set for early stopping
xgb_model.fit(
    X_train_scaled,
    y_train,
    eval_set=[(X_test_scaled, y_test)],  # Monitor test loss
    verbose=50                            # Print every 50 rounds
)

# ---- Predict on test set ----
y_pred        = xgb_model.predict(X_test_scaled)
y_pred_proba  = xgb_model.predict_proba(X_test_scaled)  # Shape: (n, 3)

# ---- Accuracy ----
accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*50}")
print(f"  TEST SET ACCURACY: {accuracy*100:.2f}%")
print(f"  Actual best rounds: {xgb_model.best_iteration}")
print(f"{'='*50}")

# ---- Classification report ----
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["W (Home Win)", "D (Draw)", "L (Home Loss)"]
))

# ---- Confusion matrix as a Plotly heatmap ----
# Confusion matrix shows WHERE the model makes mistakes.
# E.g. if many 'D' predictions are actually 'W', draws are being over-predicted.
cm = confusion_matrix(y_test, y_pred)
cm_labels = ["W (Win)", "D (Draw)", "L (Loss)"]

# Normalize rows to show percentages (easier to interpret)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_text = [
    [f"{cm_normalized[i][j]*100:.1f}%<br>({cm[i][j]})" for j in range(3)]
    for i in range(3)
]

fig_cm = go.Figure(data=go.Heatmap(
    z=cm_normalized,
    x=[f"Predicted: {l}" for l in cm_labels],
    y=[f"Actual: {l}"    for l in cm_labels],
    text=cm_text,
    texttemplate="%{text}",
    colorscale="Blues",
    showscale=True,
))

fig_cm.update_layout(
    title=f"Confusion Matrix — XGBoost (Test Accuracy: {accuracy*100:.1f}%)",
    xaxis_title="Predicted Label",
    yaxis_title="True Label",
    width=600, height=450,
)
fig_cm.show()

# ---- Baseline comparison ----
# Always predicting 'W' (most common outcome) gives what accuracy?
majority_class = y_train.mode()[0]
baseline_acc = (y_test == majority_class).mean()
lift = (accuracy - baseline_acc) / baseline_acc * 100
print(f"\nBaseline accuracy (always predict '{LABEL_MAP_INV[majority_class]}'): {baseline_acc*100:.2f}%")
print(f"Our model lift over baseline: +{lift:.1f}%")

## Feature Importance

XGBoost tracks how much each feature contributed to reducing prediction error across all 300 trees. Features with higher importance had more **split decisions** made on them.

We use `weight` importance (count of splits). You can also try `gain` (average improvement per split) by passing `importance_type='gain'` — that tends to favour fewer, high-quality splits.

**What to look for:**
- If `home_win_rate_last10` dominates → recent form drives predictions (expected)
- If `h2h_win_rate` is high → historical rivalry matters
- If `goal_diff_trend` is low → trajectory doesn't help much vs raw form

In [ ]:
# ============================================================
# Cell 5 — Feature importance chart
# ============================================================

# ---- Extract importance values ----
# get_booster().get_score() returns dict of {feature: score}
# We use 'gain' — average information gain per split (more meaningful than raw count)
importance_gain  = xgb_model.get_booster().get_score(importance_type='gain')
importance_weight = xgb_model.get_booster().get_score(importance_type='weight')

# Map feature names back (XGBoost uses f0, f1... internally)
# The XGBClassifier with sklearn API preserves feature names if we pass DataFrame
# If they come back as f0/f1/..., map them manually
feature_names = FEATURE_COLS

# Build sorted importance DataFrame
df_importance = pd.DataFrame({
    "feature":   feature_names,
    "importance_gain":   [importance_gain.get(f, importance_gain.get(f"f{i}", 0))
                          for i, f in enumerate(feature_names)],
    "importance_weight": [importance_weight.get(f, importance_weight.get(f"f{i}", 0))
                          for i, f in enumerate(feature_names)],
}).sort_values("importance_gain", ascending=True)

# Normalize to 0-100 for easier reading
df_importance["importance_pct"] = (
    df_importance["importance_gain"]
    / df_importance["importance_gain"].sum() * 100
).round(2)

# ---- Plotly horizontal bar chart ----
# Horizontal bar chart is the standard for feature importance — long names fit better
colors = [
    "#1f77b4" if "home" in f else
    "#ff7f0e" if "away" in f else
    "#2ca02c"   # H2H features in green
    for f in df_importance["feature"]
]

fig_imp = go.Figure(go.Bar(
    x=df_importance["importance_pct"],
    y=df_importance["feature"],
    orientation="h",
    marker_color=colors,
    text=[f"{v:.1f}%" for v in df_importance["importance_pct"]],
    textposition="outside",
))

fig_imp.update_layout(
    title="XGBoost Feature Importance (Gain) — Blue=Home, Orange=Away, Green=H2H",
    xaxis_title="Importance (% of total gain)",
    yaxis_title="Feature",
    height=600,
    margin=dict(l=220),
)
fig_imp.show()

print("\nTop 5 most important features:")
top5 = df_importance.sort_values("importance_pct", ascending=False).head(5)
for _, row in top5.iterrows():
    print(f"  {row['feature']:<40} {row['importance_pct']:>6.2f}%")

## Predicting 2026 World Cup Outcomes

Now we apply the trained model to the 48 WC 2026 group stage fixtures.

For each fixture, we need the same features the model was trained on. We get these by:
1. Querying the most recent feature snapshot per team from `match_features`
2. Renaming columns to match training feature names
3. Calling `predict_proba()` to get Win/Draw/Loss probabilities

The **confidence** of a prediction is `max(P(Win), P(Draw), P(Loss))`. A confidence above 0.5 means the model strongly favors one outcome.

Results are saved to the `wc2026_predictions` table in PostgreSQL — this is what the Streamlit dashboard reads in Phase 5.

In [ ]:
# ============================================================
# Cell 6 — Predict 2026 WC outcomes + save to PostgreSQL
# ============================================================
engine = create_engine(DB_URL, echo=False)

# ---- Get the LATEST feature snapshot per team ----
# For each team, we want their most recent row in match_features
# (the row that has their current form stats computed from Phase 2).
# We use DISTINCT ON (PostgreSQL-specific) to get the latest row per team.

print("Computing current form per team...")

# Latest features for each team in the HOME role
df_home_form = pd.read_sql("""
    SELECT DISTINCT ON (home_team)
        home_team                    AS team,
        home_win_rate_last10         AS win_rate_last10,
        home_draw_rate_last10        AS draw_rate_last10,
        home_avg_goals_scored_last20 AS avg_goals_scored_last20,
        home_avg_goals_conceded_last20 AS avg_goals_conceded_last20,
        home_avg_goal_diff_last10    AS avg_goal_diff_last10,
        home_goal_diff_trend         AS goal_diff_trend,
        home_ranking_proxy           AS ranking_proxy,
        home_ranking_change          AS ranking_change
    FROM match_features
    WHERE home_team IS NOT NULL
    ORDER BY home_team, date DESC    -- DISTINCT ON picks the first row per team = latest
""", engine)

# Latest features for each team in the AWAY role
df_away_form = pd.read_sql("""
    SELECT DISTINCT ON (away_team)
        away_team                    AS team,
        away_win_rate_last10         AS win_rate_last10,
        away_draw_rate_last10        AS draw_rate_last10,
        away_avg_goals_scored_last20 AS avg_goals_scored_last20,
        away_avg_goals_conceded_last20 AS avg_goals_conceded_last20,
        away_avg_goal_diff_last10    AS avg_goal_diff_last10,
        away_goal_diff_trend         AS goal_diff_trend,
        away_ranking_proxy           AS ranking_proxy,
        away_ranking_change          AS ranking_change
    FROM match_features
    WHERE away_team IS NOT NULL
    ORDER BY away_team, date DESC
""", engine)

# Combine home + away form: average the two perspectives for each team
# (A team has the same strength regardless of home/away venue)
form_cols = [
    "win_rate_last10", "draw_rate_last10",
    "avg_goals_scored_last20", "avg_goals_conceded_last20",
    "avg_goal_diff_last10", "goal_diff_trend",
    "ranking_proxy", "ranking_change"
]

# Merge home and away snapshots; average when both exist
df_team_form = df_home_form.merge(
    df_away_form, on="team", how="outer", suffixes=("_h", "_a")
)
for col in form_cols:
    df_team_form[col] = df_team_form[[f"{col}_h", f"{col}_a"]].mean(axis=1)
df_team_form = df_team_form[["team"] + form_cols].set_index("team")

print(f"Teams with form data: {len(df_team_form):,}")

# ---- Load WC 2026 fixtures ----
df_fixtures = pd.read_sql("SELECT * FROM wc2026_fixtures", engine)
print(f"Fixtures to predict: {len(df_fixtures)}")

# ---- H2H data ----
df_h2h = pd.read_sql("""
    SELECT 
        h2h_team_a_win_rate,
        h2h_total,
        -- Identify which team is 'team_a' for each h2h pair
        CASE 
            WHEN home_team < away_team THEN home_team 
            ELSE away_team 
        END AS team_a,
        CASE 
            WHEN home_team < away_team THEN away_team 
            ELSE home_team 
        END AS team_b
    FROM (
        SELECT DISTINCT ON (LEAST(home_team, away_team), GREATEST(home_team, away_team))
            home_team, away_team, h2h_team_a_win_rate, h2h_total
        FROM match_features
        ORDER BY LEAST(home_team, away_team), GREATEST(home_team, away_team), date DESC
    ) latest_h2h
""", engine)

# ---- Build inference feature matrix ----
inference_rows = []

for _, fixture in df_fixtures.iterrows():
    home = fixture["home_team"]
    away = fixture["away_team"]
    
    # Get home team's latest form (default = 0.33 win rate, neutral stats)
    if home in df_team_form.index:
        h = df_team_form.loc[home]
    else:
        print(f"  WARNING: No form data for {home} — using neutral defaults")
        h = pd.Series({c: 0.33 if "rate" in c else 0.0 for c in form_cols})
    
    if away in df_team_form.index:
        a = df_team_form.loc[away]
    else:
        print(f"  WARNING: No form data for {away} — using neutral defaults")
        a = pd.Series({c: 0.33 if "rate" in c else 0.0 for c in form_cols})
    
    # H2H lookup (normalize team pair alphabetically)
    ta = min(home, away)
    tb = max(home, away)
    h2h_row = df_h2h[(df_h2h["team_a"] == ta) & (df_h2h["team_b"] == tb)]
    
    if not h2h_row.empty:
        h2h_win_rate = h2h_row.iloc[0]["h2h_team_a_win_rate"]
        # If home team is team_a, use as-is; otherwise flip
        if home != ta:  # home team is team_b
            h2h_win_rate = 1.0 - h2h_win_rate - 0.25  # Approx away win rate
        h2h_total = h2h_row.iloc[0]["h2h_total"]
    else:
        h2h_win_rate = 0.33  # No history → equal probability
        h2h_total    = 0
    
    # Build feature row in the SAME order as FEATURE_COLS
    row = {
        "fixture_id":   fixture["fixture_id"],
        "group_name":   fixture["group_name"],
        "home_team":    home,
        "away_team":    away,
        "match_date":   fixture["match_date"],
        # Features
        "home_win_rate_last10":           h["win_rate_last10"],
        "away_win_rate_last10":           a["win_rate_last10"],
        "home_draw_rate_last10":          h["draw_rate_last10"],
        "away_draw_rate_last10":          a["draw_rate_last10"],
        "home_avg_goals_scored_last20":   h["avg_goals_scored_last20"],
        "away_avg_goals_scored_last20":   a["avg_goals_scored_last20"],
        "home_avg_goals_conceded_last20": h["avg_goals_conceded_last20"],
        "away_avg_goals_conceded_last20": a["avg_goals_conceded_last20"],
        "home_avg_goal_diff_last10":      h["avg_goal_diff_last10"],
        "away_avg_goal_diff_last10":      a["avg_goal_diff_last10"],
        "home_goal_diff_trend":           h["goal_diff_trend"],
        "away_goal_diff_trend":           a["goal_diff_trend"],
        "home_ranking_proxy":             h["ranking_proxy"],
        "away_ranking_proxy":             a["ranking_proxy"],
        "home_ranking_change":            h["ranking_change"],
        "away_ranking_change":            a["ranking_change"],
        "h2h_team_a_win_rate":            h2h_win_rate,
        "h2h_total":                      h2h_total,
    }
    inference_rows.append(row)

df_inference = pd.DataFrame(inference_rows)
print(f"\nInference rows built: {len(df_inference)}")

# ---- Scale and predict ----
X_inf = df_inference[FEATURE_COLS].fillna(0)
X_inf_scaled = scaler.transform(X_inf)  # Use SAME scaler fitted on training data

# predict_proba returns shape (n_fixtures, 3) → [P(W), P(D), P(L)]
proba = xgb_model.predict_proba(X_inf_scaled)
labels_pred = xgb_model.predict(X_inf_scaled)

# ---- Build predictions DataFrame ----
df_preds = df_inference[["fixture_id", "group_name", "home_team", "away_team", "match_date"]].copy()
df_preds["predicted_result"] = [LABEL_MAP_INV[l] for l in labels_pred]
df_preds["win_prob"]         = proba[:, 0].round(4)   # P(home win)
df_preds["draw_prob"]        = proba[:, 1].round(4)   # P(draw)
df_preds["loss_prob"]        = proba[:, 2].round(4)   # P(home loss)
df_preds["confidence"]       = proba.max(axis=1).round(4)  # Max probability = confidence
df_preds["predicted_at"]     = datetime.now().isoformat()

# ---- Save to PostgreSQL ----
df_preds.to_sql(
    name="wc2026_predictions",
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
)
print(f"\nSaved {len(df_preds)} predictions to 'wc2026_predictions' table.")

# ---- Quick preview ----
print("\nPrediction sample (sorted by confidence):")
display(
    df_preds.sort_values("confidence", ascending=False)
    [["home_team", "away_team", "group_name",
      "predicted_result", "win_prob", "draw_prob", "loss_prob", "confidence"]]
    .head(10)
    .style.format({"win_prob": "{:.1%}", "draw_prob": "{:.1%}",
                   "loss_prob": "{:.1%}", "confidence": "{:.1%}"})
)

# ---- Save model ----
import joblib

models_dir = Path("..") / "models"
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "xgb_fifa_model.pkl"
joblib.dump(xgb_model, model_path)
print(f"\nModel saved: {model_path}")

# Also save scaler and metadata (needed by dashboard and Airflow retraining)
scaler_path = models_dir / "scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"Scaler saved: {scaler_path}")

metadata = {
    "feature_cols":      FEATURE_COLS,
    "label_map":         LABEL_MAP,
    "label_map_inv":     {str(k): v for k, v in LABEL_MAP_INV.items()},
    "training_date":     datetime.now().isoformat(),
    "n_training_samples": int(len(X_train)),
    "n_test_samples":    int(len(X_test)),
    "test_accuracy":     float(accuracy),
    "best_iteration":    int(xgb_model.best_iteration),
}
with open(models_dir / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved: {models_dir / 'model_metadata.json'}")

engine.dispose()

## Top 10 Most Confident Predictions with Feature Reasoning

For each prediction, we explain **which features drove the model's decision**.

We use a simplified contribution score:
```
contribution_i = standardized_feature_value_i × feature_importance_i
```

A **positive** contribution for the predicted class means that feature pushed the model toward that prediction. This is an approximation — for full Shapley values, use the `shap` library.

**Why show this on the dashboard?** It makes predictions **trustworthy** — a user can see "Argentina predicted to win because they have 80% win rate in last 10 AND strong H2H record" rather than a black-box number.

In [ ]:
# ============================================================
# Cell 7 — Top 10 most confident predictions with feature reasoning
# ============================================================
engine = create_engine(DB_URL, echo=False)

# ---- Load predictions and feature values together ----
df_top = (
    df_preds
    .merge(df_inference[FEATURE_COLS + ["fixture_id"]], on="fixture_id")
    .sort_values("confidence", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# ---- Compute per-prediction feature contributions ----
# Standardize feature values (z-score) using training set stats
# contribution_i = z_score_i × importance_pct_i
importance_lookup = dict(zip(
    df_importance["feature"],
    df_importance["importance_pct"]
))

print("TOP 10 MOST CONFIDENT 2026 WC GROUP STAGE PREDICTIONS")
print("=" * 65)

result_icons = {"W": "WIN ", "D": "DRAW", "L": "LOSS"}

for idx, row in df_top.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    pred = row["predicted_result"]
    conf = row["confidence"]
    
    win_p  = row["win_prob"]
    draw_p = row["draw_prob"]
    loss_p = row["loss_prob"]
    
    print(f"\n#{idx+1}  {home} vs {away}  (Group {row['group_name']})")
    print(f"    Prediction: {result_icons[pred]}  |  Confidence: {conf*100:.1f}%")
    print(f"    P(Win): {win_p*100:.1f}%  P(Draw): {draw_p*100:.1f}%  P(Loss): {loss_p*100:.1f}%")
    
    # ---- Compute feature contributions for this row ----
    z_scores = {}
    for i, feat in enumerate(FEATURE_COLS):
        val = row[feat]
        z   = (val - feature_means[i]) / (feature_stds[i] + 1e-9)
        imp = importance_lookup.get(feat, 0)
        # If predicted WIN (label 0): positive z_score on win_rate features helps
        # We sign the contribution by direction alignment with the prediction
        sign = 1 if ("home" in feat and pred == "W") or \
                    ("away" not in feat and "h2h" not in feat) else -1
        z_scores[feat] = z * imp
    
    # Sort features by absolute contribution
    top_features = sorted(z_scores.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
    
    print("    Key drivers (top 5 features):")
    for feat, score in top_features:
        actual_val = row[feat]
        direction = "▲" if score > 0 else "▼"
        print(f"      {direction} {feat:<42} = {actual_val:>7.3f}")

print("\n" + "=" * 65)

# ---- Group stage predicted winners summary ----
print("\nGROUP STAGE PREDICTED WINNERS (by predicted wins count):")
print("-" * 50)

# Count predicted wins per team
home_wins = (
    df_preds[df_preds["predicted_result"] == "W"]
    .groupby("home_team").size().reset_index(name="predicted_wins")
    .rename(columns={"home_team": "team"})
)
away_wins = (
    df_preds[df_preds["predicted_result"] == "L"]
    .groupby("away_team").size().reset_index(name="predicted_wins")
    .rename(columns={"away_team": "team"})
)
all_wins = pd.concat([home_wins, away_wins]).groupby("team")["predicted_wins"].sum()
group_winners = (
    df_preds[["home_team", "away_team", "group_name"]]
    .melt(id_vars="group_name", value_name="team")
    .drop(columns="variable")
    .drop_duplicates()
    .merge(all_wins, on="team", how="left")
    .fillna(0)
    .sort_values(["group_name", "predicted_wins"], ascending=[True, False])
)
print(group_winners.to_string(index=False))

engine.dispose()
print("\nPhase 4 complete! Model trained, predictions saved, model files written.")